# System B: QLoRA Fine-Tuning Pipeline on Google Colab (Free T4 GPU)
### Project: Agentic RAG with Evaluation Harness — College Student Handbook
**Student**: 3rd Year B.Tech CS & AI | **Target Model**: `Qwen/Qwen2.5-1.5B-Instruct`

---
## Theoretical Overview (What to explain in Viva)
1. **What is QLoRA?** Efficient fine-tuning combining 4-bit NormalFloat (NF4) quantization of frozen base model weights with Low-Rank Adaptation (LoRA) trainable adapter matrices ($W = W_0 + \Delta W$, where $\Delta W = B \times A$, $r=16$).\n2. **Memory Footprint**: FP16 base model = ~3.2 GB. Under 4-bit NF4 = ~1.1 GB. Trainable parameters = ~18.4M (~1.2% of model). Peak VRAM during training on T4 = ~4.2 GB.
3. **Double Quantization**: Quantizes the quantization constants themselves, saving an additional 0.37 bits per parameter.
4. **Paged Optimizers**: Uses CUDA Unified Memory to page memory between GPU and CPU when allocation spikes occur during gradient backpropagation.

In [ ]:
# Step 1: Verify GPU Allocation (Expected: NVIDIA Tesla T4 16GB)
!nvidia-smi

In [ ]:
# Step 2: Install required libraries for QLoRA and PEFT
!pip install -q -U torch transformers peft trl bitsandbytes accelerate datasets

In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")

### Step 3: Upload and Load Dataset
Upload `data/processed/sft_train.jsonl` and `data/processed/sft_val.jsonl` from your project workspace.

In [ ]:
# Upload datasets via Colab files UI or run this cell
from google.colab import files
uploaded = files.upload() # Select sft_train.jsonl and sft_val.jsonl

In [ ]:
# Load training dataset into HuggingFace dataset format
dataset = load_dataset('json', data_files={'train': 'sft_train.jsonl', 'validation': 'sft_val.jsonl'})
print(dataset)
print("Sample prompt:")
print(dataset['train'][0]['text'][:300])

### Step 4: Configure 4-bit NormalFloat (NF4) Quantization

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
print(model)

### Step 5: Configure LoRA Adapters (r=16, alpha=32)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

### Step 6: Train with SFTTrainer (Supervised Fine-Tuning)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_qlora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

### Step 7: Save Adapter Weights & Export for Local Deployment

In [ ]:
ADAPTER_DIR = "./qlora_handbook_adapter"
peft_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved LoRA adapter weights to: {ADAPTER_DIR}")

# Create downloadable zip archive
!zip -r qlora_handbook_adapter.zip ./qlora_handbook_adapter
from google.colab import files
files.download('qlora_handbook_adapter.zip')
print("Download initiated! Unzip this into your agentic-rag-eval/adapters/ directory.")

### Step 8: Qualitative Inference Verification (Sanity Check)

In [ ]:
def test_inference(question: str):
    prompt = f"<|im_start|>system\nYou are an advisor at Apex Institute of Technology.<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = peft_model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}\nA: {response}\n" + "-"*60)

test_inference("What is the medical attendance condonation fee?")
test_inference("What is the curfew timing for hostels on weekdays?")